# 02 · Previsão mensal por variável com LightGBM

Execute o Notebook 01 primeiro. Ajuste um modelo LightGBM para cada variável
dinâmica e horizonte mensal. Compare o MAE do recorte cronológico com a
persistência do último valor e publique o método com menor erro.

A geração em lote recarrega o pickle produzido pela pipeline no mesmo ambiente.
A aplicação consome as previsões publicadas em Parquet.

## Configurar caminhos

In [1]:
import sys
from pathlib import Path

import pandas as pd

ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src/feynmans_legacy/analog_study.py").exists()
)
sys.path.insert(0, str(ROOT / 'src'))
TERRA = ROOT / 'data/terra'
RUN = TERRA / 'analise_analogos'
RUN.mkdir(parents=True, exist_ok=True)

## Definir alvos, horizontes e amostra de treino

O ajuste e a validação cronológica usam uma amostra reproduzível de locais.
A inferência usa todos os locais com defasagens suficientes no mês de corte.

In [2]:
import json
import pickle

from feynmans_legacy.analog_study import LAGS, publish_forecasts, train_forecasts

HORIZONTES = [1, 2, 3]
LOCAIS_TREINO = 400
SEMENTE = 42
selection = json.loads((RUN / "selecao.json").read_text(encoding="utf-8"))
TARGETS = selection["alvos_previsao"]
terra_path = TERRA / "terra.parquet"
locations = pd.read_parquet(RUN / "coordenadas.parquet", columns=["location_id"])
sample = locations["location_id"].sort_values().sample(
    n=min(LOCAIS_TREINO, len(locations)), random_state=SEMENTE
)
history = pd.read_parquet(
    terra_path, columns=["location_id", "anomes", *TARGETS],
    filters=[("location_id", "in", sample.tolist())],
)
sample_size = history["location_id"].nunique()
print(f"Treino com {sample_size:,} locais e {len(TARGETS)} variáveis dinâmicas")

Treino com 400 locais e 9 variáveis dinâmicas


## Treinar LightGBM e comparar com a persistência

In [3]:
bundle, metrics = train_forecasts(history, TARGETS, HORIZONTES)
metrics["locais_amostra"] = history["location_id"].nunique()
metrics["semente_amostra"] = SEMENTE
if not bundle["modelos"]:
    raise ValueError("Histórico insuficiente para publicar as previsões configuradas.")

## Salvar o modelo e as métricas

In [4]:
model_dir = RUN / "modelos"
model_dir.mkdir(exist_ok=True)
artifact = model_dir / "previsao_mensal.pkl"
with artifact.open("wb") as file:
    pickle.dump(bundle, file, protocol=pickle.HIGHEST_PROTOCOL)
metrics.to_parquet(RUN / "metricas_previsao.parquet", index=False)


## Conferir o erro por horizonte

In [5]:
print(metrics.to_string(index=False))

                                          variavel  horizonte   status  n_validacao  mae_modelo  mae_persistencia  locais_amostra  semente_amostra
                              temperatura__ar_2m_c          1 lightgbm         6800    1.502457          3.986471             400               42
                              temperatura__ar_2m_c          2 lightgbm         6400    2.037692          7.345203             400               42
                              temperatura__ar_2m_c          3 lightgbm         6000    2.594905         10.414683             400               42
            temperatura__lst_amplitude_dia_noite_c          1 lightgbm         7078    1.326804          1.691509             400               42
            temperatura__lst_amplitude_dia_noite_c          2 lightgbm         6676    1.846681          2.406561             400               42
            temperatura__lst_amplitude_dia_noite_c          3 lightgbm         6276    2.284543          2.968069     

A inferência em lote lê apenas os meses necessários no Parquet observado.
Ao atualizar o mês de corte, regenere o artefato e revise as métricas.
O mapa lê o Parquet publicado.

## Recarregar o modelo e publicar as previsões

In [6]:
cutoff = pd.Period(bundle["origem"], freq="M")
lag_months = [int((cutoff - lag).strftime("%Y%m")) for lag in LAGS]
recent = pd.read_parquet(
    terra_path, columns=["location_id", "anomes", *TARGETS],
    filters=[("anomes", "in", lag_months)],
)
forecasts = publish_forecasts(recent, artifact)
if forecasts.empty:
    raise ValueError("Nenhum local possui as entradas necessárias no mês de corte.")
forecasts.to_parquet(RUN / "previsoes.parquet", index=False)
print(f"{len(forecasts):,} previsões para {forecasts['location_id'].nunique():,} locais")

2,002,302 previsões para 85,726 locais


## Conferir os valores previstos

In [7]:
print(forecasts.head().to_string(index=False))

location_id  anomes             variavel  valor_previsto
    T000001  202609 temperatura__ar_2m_c       -8.793247
    T000002  202609 temperatura__ar_2m_c      -10.123420
    T000003  202609 temperatura__ar_2m_c      -10.123420
    T000004  202609 temperatura__ar_2m_c      -10.123420
    T000005  202609 temperatura__ar_2m_c      -10.123420
